# Inspector: del archivo al borrador de mapping (P0-B1)
Todo es local. El Inspector perfila el archivo, analiza **en tu máquina** los valores de la URL y propone un mapping que **tú revisas y apruebas**. Los ejemplos salen enmascarados.

In [ ]:
%load_ext autoreload
%autoreload 2
from dfir_copilot.profiling import inspect_source

RUTA = "/workspace/data/raw/three_months.csv"   # CSV, TSV, JSON/NDJSON o Parquet (también .gz)
IDIOMA = "es"

%time borrador = inspect_source(RUTA, lang=IDIOMA)
print(borrador.render())

## Revisa el borrador
Guárdalo, ábrelo, corrige lo que haga falta (zona horaria, nombres, roles) y **ese** archivo es el mapping aprobado.

In [ ]:
RUTA_BORRADOR = "/workspace/data/mappings_borradores/three_months.yaml"
borrador.save(RUTA_BORRADOR)
print(open(RUTA_BORRADOR, encoding="utf-8").read())

## Ingerir en un caso nuevo con el mapping aprobado
Un caso = un dataset: usa un identificador nuevo.

In [ ]:
from dfir_copilot.cases import CaseWorkspace

CASO = "IDOR-INVOICES-2020Q4-INSPECTOR"
ws = CaseWorkspace.open_or_create(CASO, root="/workspace/data/cases", analyst="eder", lang=IDIOMA)
ws.add_raw(RUTA, link=True)
if not ws.meta["dataset"]:
    manifest = ws.ingest(RUTA, RUTA_BORRADOR)
    print(manifest["output"]["rows"], "filas |", manifest["roles"], "|", manifest["warnings"])
else:
    print("ya ingerido:", ws.meta["dataset"]["rows"], "filas")

In [ ]:
print(ws.verify().render(IDIOMA))